In [1]:
# ============================================================================
# BLOCK 1: DATA LOADING - ROMANIAN
# Input: ro_comments.csv
# Output: 01_loaded_data_ro.pkl
# ============================================================================

# CELL 1: Load configuration from setup notebook
# (Run 00_setup_and_config.ipynb first)
%run 00_setup_and_config.ipynb

c:\Users\andre\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Libraries loaded successfully
✓ PyTorch device: CPU
✓ Timestamp: 2026-05-29 04:08:30
✓ Directory structure verified
✓ Base directory: c:\Users\andre\OneDrive\Desktop\2_Disertatie_RIPEMC_D.Flanja\Quantitative_Python\YT_Analysis
✓ Model configuration loaded
  - Polish sentiment: eevvgg/bert-polish-sentiment-politics
  - Romanian sentiment: readerbench/ro-sentiment
  - Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  - BERTopic topics: 8
  - Batch size: 8
✓ Checkpoint utility functions loaded
✓ Text cleaning functions loaded
✓ Romanian stopwords: 385 words
✓ Polish stopwords: 511 words
✓ Visualization utility functions loaded
✓ Sentiment prediction function loaded
✓ SETUP AND CONFIGURATION COMPLETE

Ready for analysis. Next steps:
  1. Run Polish pipeline: 01_data_loading_pl.ipynb → 06_engagement_metrics_pl.ipynb
  2. Run Romanian pipeline: 01_data_loading_ro.ipynb → 06_engagement_metrics_ro.ipynb
  3. (Optional) Run comparative analysis: 08_comparative_ana

In [2]:
# CELL 2: Load raw data
print('='*70)
print('BLOCK 1: DATA LOADING - ROMANIAN')
print('='*70)

if check_checkpoint_exists('ro', '01_loaded_data'):
    df_ro = load_checkpoint('ro', '01_loaded_data')
else:
    raw_file = RAW_DIR / 'romanian_comments.csv'
    
    if not raw_file.exists():
        raise FileNotFoundError(f'Romanian comments file not found: {raw_file}')
    
    print(f'\nLoading: {raw_file}')
    df_ro = pd.read_csv(raw_file)
    
    print(f'\n✓ Loaded {len(df_ro):,} rows × {len(df_ro.columns)} columns')
    
    def to_snake_case(col_name: str) -> str:
        col = str(col_name).strip().lower()
        col = re.sub(r'[^a-z0-9]+', '_', col)
        col = re.sub(r'_+', '_', col).strip('_')
        return col
    
    df_ro.columns = [to_snake_case(c) for c in df_ro.columns]
    
    print(f'\nColumns after standardization:')
    print(df_ro.columns.tolist())
    
    for col in ['comment_likes', 'replies']:
        if col in df_ro.columns:
            df_ro[col] = pd.to_numeric(df_ro[col], errors='coerce')
    
    if 'comment_date' in df_ro.columns:
        df_ro['comment_date'] = pd.to_datetime(df_ro['comment_date'], errors='coerce', utc=True)
    
    if 'is_creator' in df_ro.columns:
        df_ro = df_ro.drop(columns=['is_creator'])
        print('\n✓ Dropped is_creator column')
    
    save_checkpoint(df_ro, 'ro', '01_loaded_data')
    update_pipeline_status('ro', 1, 'completed')

BLOCK 1: DATA LOADING - ROMANIAN

Loading: data\raw\romanian_comments.csv

✓ Loaded 6,790 rows × 7 columns

Columns after standardization:
['video_id', 'author_name', 'comment_text', 'comment_date', 'comment_likes', 'replies', 'is_creator']

✓ Dropped is_creator column
✓ Checkpoint saved: 01_loaded_data_ro.pkl
✓ Pipeline status updated: ro - Block 1 - completed


In [3]:
# CELL 3: Data audit
print('\n' + '='*70)
print('DATA AUDIT - ROMANIAN')
print('='*70)

audit_summary = {
    'Total Rows': len(df_ro),
    'Total Columns': len(df_ro.columns),
    'Exact Duplicates': int(df_ro.duplicated().sum()),
    'Total Missing Cells': int(df_ro.isna().sum().sum()),
    'Rows with Any Missing': int(df_ro.isna().any(axis=1).sum())
}

if 'video_id' in df_ro.columns:
    audit_summary['Unique Videos'] = int(df_ro['video_id'].nunique(dropna=True))
if 'author_name' in df_ro.columns:
    audit_summary['Unique Authors'] = int(df_ro['author_name'].nunique(dropna=True))

for key, value in audit_summary.items():
    print(f'  {key}: {value:,}' if isinstance(value, int) else f'  {key}: {value}')

print('\nDataset Preview (first 5 rows):')
display(df_ro.head(5))

print('\n' + '='*70)
print('✓ BLOCK 1 COMPLETE - ROMANIAN DATA LOADED')
print('='*70)



DATA AUDIT - ROMANIAN
  Total Rows: 6,790
  Total Columns: 6
  Exact Duplicates: 0
  Total Missing Cells: 3
  Rows with Any Missing: 3
  Unique Videos: 14
  Unique Authors: 5,545

Dataset Preview (first 5 rows):


,video_id,author_name,comment_text,comment_date,comment_likes,replies
0,d9U66yU8dYE,@CristiSiRalu,"Prea putin se vorbeste ca omul asta merge ore in sir, chiar de doua ori pe zi la dezbateri expun...",2025-05-14 17:25:24+00:00,5199,340
1,d9U66yU8dYE,@ClaudeFrost,"Săracul Nicușor, se vede pe el cât este de obosit mai ales din cauza faptului că mai bine de o s...",2025-05-14 15:56:34+00:00,3055,142
2,d9U66yU8dYE,@d.gabriele230,Am votat in primul tur Simion. Tind sa-mi schimb votul dupa acest podcast! Felicitari Domnului N...,2025-05-14 17:18:09+00:00,2876,191
3,d9U66yU8dYE,@lorentzo96,"Daca tot vorbim de empatie, nu ai cum sa nu apreciez stresul si oboseala prin care trece acest o...",2025-05-14 18:20:30+00:00,1389,42
4,d9U66yU8dYE,@laurapuescu3155,"Nu credeam că voi spune asta vreodată, dar a ajuns să îmi facă plăcere să ascult o dezbatere pre...",2025-05-14 16:57:30+00:00,1173,42



✓ BLOCK 1 COMPLETE - ROMANIAN DATA LOADED
